# Standards-Based Grading pipeline — runnable demo

This notebook runs the complete SBG starter-kit pipeline on the bundled
synthetic sample data (30 fictitious students): platform exports →
canonical scores → standards achieved → individual student reports.

Based on the MAT188 (University of Toronto) system by Camelia Karimianpour
and collaborators; pipeline adapted from
[dtxe/mat188_learning_standards](https://github.com/dtxe/mat188_learning_standards).


In [ ]:
# Get the template (set REPO_URL after you publish it on GitHub).
import os
REPO_URL = ''   # e.g. 'https://github.com/<you>/sbg-template'
if os.path.exists('run_pipeline.py'):
    ROOT = '.'                       # running inside the repo already
elif REPO_URL:
    !git clone -q {REPO_URL} sbg-template
    ROOT = 'sbg-template'
else:
    raise SystemExit('Upload the sbg-template folder or set REPO_URL')
%cd {ROOT}
!pip -q install pandas pyyaml openpyxl


## 1. Look at the inputs

`standards.csv` holds the learning standards; the lookup table connects
each standard × modality to the score keys that demonstrate it
(`2|ww1-1,ww1-2,ww1-3` = at least 2 of these 3 correct).


In [ ]:
import pandas as pd
display(pd.read_csv('standards/standards.csv'))
display(pd.read_csv('lookup/standards_lookup_table.csv').fillna(''))


In [ ]:
# A raw WeBWorK export (note the metadata rows — the converter handles them)
print(open('sample_data/webwork/ww1.csv').read()[:600])


## 2. Run the pipeline


In [ ]:
!python run_pipeline.py config.yml


## 3. Inspect the results


In [ ]:
sa = pd.read_csv('output/standards_achieved.csv', header=[0,1], index_col=0)
sa['summary'].head(10)


In [ ]:
# Full detail for one student: 1 = demonstrated, 0 = assessed not yet
# demonstrated, NaN = not yet assessed in that modality
sa.drop(columns='summary', level=0).loc['student01'].unstack('modality')


In [ ]:
# Render that student's report
from IPython.display import HTML
HTML(open('output/reports_html/student01.html').read())


## 4. Resubmissions: only eventual mastery matters

Append the resubmission export and re-evaluate — demonstrations can only
be added, never lost.


In [ ]:
import yaml, sys, os
sys.path.insert(0, 'scripts')
import evaluate_standards
cfg = yaml.safe_load(open('config.yml'))
cfg['tutorial_files'] = ['sample_data/tutorials/tutorial_scores.csv',
                         'sample_data/tutorials/resubmission_scores.csv']
cfg['output_dir'] = 'output_resub'
os.makedirs('output_resub', exist_ok=True)
evaluate_standards.run(cfg)
before = sa['tutorial']
after = pd.read_csv('output_resub/standards_achieved.csv', header=[0,1], index_col=0)['tutorial']
print('new demonstrations:', int(((before != 1) & (after == 1)).sum().sum()))
print('regressions (must be 0):', int(((before == 1) & (after == 0)).sum().sum()))


## 5. Adapt it

1. Edit `standards/standards.csv` with your standards.
2. Label your questions with score keys; fill `lookup/standards_lookup_table.csv`.
3. Point `config.yml` at your real exports.

See `docs/` for the precise data formats and the labeling grammar.
